In [ ]:
from probill.agents import TeachableMcpAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools
from autogen_agentchat.ui import Console
from autogen_core import Component
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json

base_model = OpenAIChatCompletionClient(
    model="qwen2.5-coder:32b-instruct-q5_0",
    api_key="sk-proj-1234567890",
    base_url="http://10.0.40.49:11434/v1",
    model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": "unknown",
    "structured_output": True
    },
    temperature=0.0,
    top_p=1.0      
)

server_params=StdioServerParams(
    command="mcp-kg", 
    args=[]
)

mode = "replay" #replay

client = ChatCompletionClientRecorder(
    client=base_model,
    mode=mode,
    session_file_path="./sessions/session.json",
)

print(client.session_file_path)

agent = TeachableMcpAgent(
    name="oncology_knowledge_searcher",
    model_client=client,
    description="A Cypher script and oncology domain expert. Assist the user by generating the Cypher script to search the oncology knowledge graph.",
    server_params=server_params,
    model_client_stream=False,
    system_message="You are a Cypher script and oncology domain expert. Assist the user by generating the Cypher script to search the oncology knowledge graph. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

def export_component(c: Component)->Dict:
    try:
        _component = c.dump_component()
        _component.label = _component.config["name"]
        _component.description = _component.config["description"]
        return _component
    except Exception as e:
        return None


In [ ]:
task_description = "Search for all regimens of a conditions contains 'ERBB2 Breast cancer'"
cypher_script = """MATCH (c:Condition)-[:HAS_ACCEPTED_USE]-(r:Regimen) 
WHERE lower(c.concept_name) CONTAINS lower("ERBB2 Breast cancer")
RETURN c.concept_name,r.concept_name"""
teachable_task = f"You can use the Cypher script below to {task_description}.\n{cypher_script}"

task = "Search for all regimens for ER Breast cancer"

result = await Console(agent.run_stream(task=task), output_stats=True)

In [ ]:
client.finalize()
await agent.close()
await client.close()
# print(agent.dump_component().model_dump_json(indent=2))
# print(export_component(agent).model_dump_json(indent=2))
